# Baseline Determinization UCT MCTS for Carcassonne

This notebook demonstrates a maintainable architecture where all simulator, agent, MCTS, and evaluation logic lives in the `carc_rl` package (`src/`) and the notebook only orchestrates experiments.

**Determinization approach:** each MCTS simulation clones the state and shuffles the remaining deck in that clone. This approximates hidden-information sampling while preserving the currently visible `next_tile`.  
**Engine mapping:** the adapter wraps the WingedSheep engine API and isolates engine-specific details in `engine_adapter.py`.

In [ ]:
%pip install -e engine
%pip install -e .

In [ ]:
from carc_rl import CarcassonneSim, RandomAgent, GreedyAgent, MCTSAgent
from carc_rl.eval import play_game, run_match
import random

In [ ]:
sim = CarcassonneSim(players=2)
state = sim.reset(seed=123)
rng = random.Random(123)
agent = RandomAgent()

print('Initial:', sim.render_text(state))
for i in range(6):
    action = agent.select_action(sim, state, rng)
    state = sim.step(state, action, rng)
    print(f'Move {i+1}:', sim.render_text(state))

In [ ]:
!pytest -q

In [ ]:
sim = CarcassonneSim(players=2)
results = {}

results['Random vs Random'] = run_match(sim, RandomAgent(), RandomAgent(), n_games=3, seed=0, max_moves=80)
results['Greedy vs Random'] = run_match(sim, GreedyAgent(), RandomAgent(), n_games=3, seed=100, max_moves=80)
results['MCTS(200) vs Greedy'] = run_match(sim, MCTSAgent(n_simulations=200, seed=1), GreedyAgent(), n_games=1, seed=200, max_moves=12)
results['MCTS(1000) vs Greedy'] = run_match(sim, MCTSAgent(n_simulations=1000, seed=1), GreedyAgent(), n_games=1, seed=300, max_moves=5)

results